# Qwen2.5-1.5B-Instruct dans NeuroDSL

Charge le vrai checkpoint HuggingFace `Qwen/Qwen2.5-1.5B-Instruct` (chargé une fois, converti au format natif NeuroDSL) dans un `NeuroGraph`, et propose une fonction `chat(prompt)` pour lui parler directement depuis ce notebook.

**Ce qui est déjà vérifié avant ce notebook** (voir `notebook/qwen2_parity_check.jl` et le rapport de session) : les logits de ce graphe concordent avec le modèle HuggingFace de référence à la précision flottante (écart absolu max mesuré : `3.7e-5`), le top-1/top-5 et une continuation gloutonne de 8 tokens sont identiques token pour token sur plusieurs prompts.

**Portée livrée : multi-tours.** L'historique de conversation (`HISTORY`) est réellement réutilisé d'un appel à `chat(...)` au suivant (gabarit ChatML réel). `reset_chat!()` vide l'historique.

**Deux correctifs de performance appliqués et vérifiés ce soir, effet COMBINÉ mesuré directement dans ce notebook (cellules §5) :**
1. **Préfixe en un seul passage batché** (`NeuroDSL.prime_kv_cache_from_prefix!`) au lieu d'un remplissage du cache token par token (bug initial : ~19-28s pour un préfixe de 36 tokens). Vérifié correct : `notebook/kv_cache_prefix_prime_qwen_gate.jl`, écart max `5.6e-5` vs remplissage séquentiel.
2. **Serveur tokenizer persistant** (`qwen2_tokenizer_helper.py --serve`) au lieu de relancer Python + réimporter `transformers` + recharger le tokenizer à CHAQUE appel (~13-14s de coût fixe par appel, indépendant du cache KV). Vérifié correct : `notebook/kv_cache_tokenizer_persistent_check_run.log`, résultats identiques au mode un-coup sur encode_chat/decode/eos_id.

**Résultat combiné, mesuré dans CE notebook (§5, même prompt "What is the capital of Egypt?", 3 appels dans la même session) :**

| Appel | Temps total | Contexte |
|---|---|---|
| Cellule 8 (1er appel de la session) | **14.2s** | JIT CUDA froid + tokenizer froid (les deux coûts uniques payés une fois) |
| Appel A (§5, même session, plus tard) | **4.2s** | tout déjà chaud |
| Appel B (§5, répétition) | **4.1s** | stable |

Soit **~3.4x plus rapide** dès le deuxième message, et stable ensuite -- gain réel, visible sur le temps total perçu par l'utilisateur, pas seulement dans une mesure de calcul isolée. Le premier message d'une session reste plus lent (compilation JIT + démarrage du tokenizer, tous deux payés une seule fois) ; c'est attendu et documenté, pas un bug.

**Portée précise du cache (pour ne pas surclaimer) :** le cache KV est reconstruit à neuf à CHAQUE appel à `chat(...)` à partir de l'historique ChatML complet -- il ne persiste PAS encore D'UN appel à L'AUTRE (économiserait le retraitement de l'historique des tours précédents ; pas construit, pas validé). Le serveur tokenizer, lui, persiste bien d'un appel à l'autre (c'est le correctif ci-dessus).

**Historique complet des bugs trouvés et corrigés ce soir** (pour ne pas les répéter) : cache rempli token par token pour le préfixe (corrigé -> passage batché) ; partage de namespace entre le graphe caché et le graphe de recalcul complet, qui faisait ré-exécuter incidemment des nœuds à état lors d'un `demand!` sans rapport (corrigé -> `copy_params_to_namespace!`, deux namespaces séparés) ; tokenizer relancé à froid à chaque appel (corrigé -> serveur persistant). Les trois ont été trouvés en mesurant, pas en supposant.

**Tokenizer : aucune nouvelle dépendance Julia**, dans les deux modes (`qwen2_tokenizer_helper.py`, environnement conda isolé `neurodsl_llm_check` -- `transformers`/`torch` jamais installés dans `base`).

## 1. Charger le graphe (checkpoint natif NeuroDSL, déjà validé)

In [ ]:
using NeuroDSL, JSON

const MODEL_DIR = joinpath(@__DIR__, "qwen2.5-1.5b-instruct")
const DIM, N_LAYERS, N_HEADS, N_KV_HEADS, HIDDEN_DIM, VOCAB_SIZE = 1536, 28, 12, 2, 8960, 151936
const ROPE_THETA, RMS_EPS = 1_000_000.0, 1e-6

println("Construction du graphe de chargement (recalcul complet -- sert DEUX rôles : source de poids pour copy_params_to_namespace!, ET passage avant batché du préfixe, voir cell 3)...")
dev = NeuroDSL.Backend.CUDADevice()
const ns_load = :qwen2_load   # graphe "chargeur" ET passage avant batché du préfixe (recalcul complet, un seul appel par tour)
const ns_chat = :qwen2_chat   # graphe de PRODUCTION pour la génération : décodage incrémental avec cache KV, un token à la fois
g = NeuroDSL.NeuroGraph(namespace=ns_load, device=dev)
NeuroDSL.set!(g, :token_ids, ones(Int, 8); atom_type=NeuroDSL.Datom, namespace=ns_load)
tok_emb = NeuroDSL.Embedding(VOCAB_SIZE, DIM)(g, :token_ids, :tok; namespace=ns_load)
out_sym = NeuroDSL.LlamaModel(N_LAYERS, DIM, N_HEADS, HIDDEN_DIM;
                               batched_attn=true, n_kv_heads=N_KV_HEADS,
                               qkv_bias=true, use_rope=true, rope_theta=ROPE_THETA)(g, tok_emb; namespace=ns_load)
final_norm = NeuroDSL.LayerNorm(DIM; eps=RMS_EPS)(g, out_sym, :final_norm; namespace=ns_load)
const logits_load = NeuroDSL.Linear(DIM, VOCAB_SIZE, bias=false)(g, final_norm, :lm_head; namespace=ns_load)
# `logits_load` capturé (contrairement à la version précédente) : cell 3 en a
# besoin pour faire le passage avant BATCHÉ du préfixe -- voir plus bas.

println("Chargement des poids (qwen2_neurodsl.json/.bin, format natif -- pas de re-parsing du .safetensors)...")
NeuroDSL.load_graph!(g, ns_load, joinpath(MODEL_DIR, "qwen2_neurodsl"); overwrite=true)

# Deux namespaces séparés, PAS un seul partagé -- `demand!` exécute tout nœud
# invalide qu'il rencontre dans SON namespace avant sa cible, pas seulement les
# ancêtres de la cible (voir NeuroDSL.copy_params_to_namespace! pour le détail) ;
# un namespace unique a fait planter la porte de parité (cache KV corrompu par
# un `demand!` sur un tout autre nœud) avant d'être corrigé ainsi. `ns_chat` ne
# contient QUE le graphe de décodage incrémental -- aucun nœud de recalcul
# complet à recalculer par erreur, donc le gain de vitesse du cache est réel,
# pas masqué par un travail redondant.
println("Construction du graphe de décodage incrémental avec cache KV (namespace :$ns_chat)...")
n_copied = NeuroDSL.copy_params_to_namespace!(g, ns_load, ns_chat)
const dec_logits = NeuroDSL.build_cached_decode_graph!(g;
    n_layers=N_LAYERS, dim=DIM, n_heads=N_HEADS, hidden_dim=HIDDEN_DIM, vocab_size=VOCAB_SIZE,
    n_kv_heads=N_KV_HEADS, qkv_bias=true, use_rope=true, rope_theta=ROPE_THETA, namespace=ns_chat)
println("Prêt -- $n_copied paramètres partagés (par valeur). ns_load : passage batché du préfixe. ns_chat : cache KV pour la génération.")

## 2. Pont tokenizer (process externe, environnement conda isolé, zéro nouvelle dépendance Julia)

In [ ]:
const PYTHON_ENV = raw"C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe"
const TOKENIZER_HELPER = joinpath(@__DIR__, "qwen2_tokenizer_helper.py")

# Serveur persistant (correctif du 2026-07-29, voir cell 0) : `transformers` +
# le tokenizer ne sont chargés QU'UNE FOIS, au démarrage de ce process, qui
# reste ensuite vivant pour toute la durée de la session -- au lieu de
# relancer un interpréteur Python + réimporter `transformers` + recharger le
# tokenizer depuis le disque à CHAQUE appel (`encode_chat`, `decode_ids`),
# comme le faisait la version précédente (~13-14s de coût fixe PAR appel,
# mesuré ce soir, indépendant du cache KV -- voir kv_cache_chat_timing_probe_run2.log
# et kv_cache_tokenizer_persistent_check_run.log pour la vérification de
# correction : mêmes résultats byte pour byte que le mode un-coup historique
# sur eos_id/encode_chat/decode, sur plusieurs historiques réels).
println("Démarrage du serveur tokenizer persistant (transformers + tokenizer chargés UNE fois)...")
const TOKENIZER_PROC = open(`$PYTHON_ENV -u $TOKENIZER_HELPER --serve`, "r+")
const _tok_ready = JSON.parse(readline(TOKENIZER_PROC))
println("Serveur tokenizer prêt : ", _tok_ready)

function _call_helper(req::Dict)
    println(TOKENIZER_PROC, JSON.json(req))
    flush(TOKENIZER_PROC)
    return JSON.parse(readline(TOKENIZER_PROC))
end

"""Encode l'historique de messages via le VRAI gabarit ChatML du tokenizer
(`apply_chat_template`, add_generation_prompt=true) -- pas une réimplémentation
manuelle de la chaîne `<|im_start|>...`."""
encode_chat(messages) = Int.(_call_helper(Dict("action"=>"encode_chat", "messages"=>messages))["ids"])

"""Décode une suite d'IDs (0-indexés, convention HuggingFace) en texte lisible,
en omettant les tokens spéciaux (`<|im_end|>` etc.)."""
decode_ids(ids::Vector{Int}) = _call_helper(Dict("action"=>"decode", "ids"=>ids))["text"]

const EOS_ID = _call_helper(Dict("action"=>"eos_id"))["id"]
println("Token de fin de tour <|im_end|> = ", EOS_ID, " -- confirmé depuis le tokenizer réel, pas codé en dur à l'aveugle.")

## 3. Génération avec cache KV (préfixe batché + décodage incrémental) + `chat(prompt)` multi-tours

Deux phases, PAS une seule boucle uniforme (correctif du 2026-07-29 -- voir cell 0) :
1. **Préfixe** (historique ChatML ré-encodé, plusieurs dizaines de tokens) : UN SEUL passage avant batché sur `ns_load` (recalcul complet, exactement comme n'importe quel forward pass), qui amorce directement le cache KV de `ns_chat` via `NeuroDSL.prime_kv_cache_from_prefix!` -- pas de boucle token par token pour cette partie.
2. **Génération** (les nouveaux tokens de la réponse, un par un -- chacun dépend forcément du précédent) : `set!` + `invalidate_all!` + `demand!` par token sur `ns_chat`, le cache porte l'historique.

La frontière préfixe→génération est invisible pour le cache lui-même (le mécanisme d'amorçage produit EXACTEMENT le même état de cache qu'un remplissage token par token -- vérifié par `notebook/kv_cache_prefix_prime_qwen_gate.jl`, écart max `5.6e-5`, continuation identique sur 5 pas). La boucle de génération s'arrête toujours sur le VRAI token `<|im_end|>` (`EOS_ID`) plutôt qu'après un nombre fixe de tokens.

In [ ]:
"""Avance le cache KV d'UN token (`tok0`, 0-indexé HuggingFace) à la position
`cur_step` (1-indexée) et retourne les logits `(vocab,)` qui en résultent --
la prédiction du token SUIVANT `tok0`. Réservé aux tokens GÉNÉRÉS un par un
(chacun dépend du précédent) -- PAS au préfixe, qui est traité en un seul
passage batché ci-dessous (voir cell 0 -- correctif de performance du
2026-07-29)."""
function cached_step!(g, tok0::Int, cur_step::Int)
    NeuroDSL.set!(g, :dec_token_id, [tok0 + 1]; atom_type=NeuroDSL.Datom, namespace=ns_chat)
    NeuroDSL.set!(g, :dec_cur_step, Float32[cur_step]; namespace=ns_chat)
    NeuroDSL.set!(g, :dec_pos, Float32[cur_step-1]; namespace=ns_chat)
    NeuroDSL.invalidate_all!(g; namespace=ns_chat)
    return Array(NeuroDSL.demand!(g, dec_logits; namespace=ns_chat))[1, :]
end

const HISTORY = Dict{String,Any}[]  # conversation multi-tours : [{"role"=>..,"content"=>..}, ...]

"""
    chat(prompt::String; max_new_tokens=100, verbose=true) -> String

Ajoute `prompt` comme tour utilisateur à `HISTORY`, encode TOUT l'historique
via le gabarit ChatML réel, traite ce préfixe en UN SEUL passage avant batché
(`ns_load`, recalcul complet -- comme n'importe quel forward pass), amorce le
cache KV (`ns_chat`) avec le résultat via `NeuroDSL.prime_kv_cache_from_prefix!`,
puis génère la réponse de l'assistant token par token via le cache (gloutonne,
arrêt sur `<|im_end|>` ou `max_new_tokens`), l'ajoute à `HISTORY` à son tour,
et retourne le texte décodé.
"""
function chat(prompt::AbstractString; max_new_tokens::Int=100, verbose::Bool=true)
    push!(HISTORY, Dict("role"=>"user", "content"=>String(prompt)))
    ids0 = encode_chat(HISTORY)   # 0-indexé (HuggingFace)
    prefix = ids0 .+ 1             # 1-indexé pour NeuroDSL::Embedding

    gen0 = Int[]
    stopped_on_eos = false
    t0 = time()

    # -- Préfixe : UN passage avant batché (ns_load), PAS une boucle token
    # par token -- c'est exactement ce que corrige `prime_kv_cache_from_prefix!`
    # par rapport à la première version de ce notebook (~0.5s PAR TOKEN du
    # préfixe -> ~0.5s pour TOUT le préfixe, quelle que soit sa longueur).
    NeuroDSL.set!(g, :token_ids, prefix; atom_type=NeuroDSL.Datom, namespace=ns_load)
    NeuroDSL.invalidate_all!(g; namespace=ns_load)
    prefix_out = Array(NeuroDSL.demand!(g, logits_load; namespace=ns_load))
    logits_row = Float32.(prefix_out[end, :])   # prédiction du 1er token de la réponse
    NeuroDSL.prime_kv_cache_from_prefix!(g; src_ns=ns_load, dst_ns=ns_chat,
        n_layers=N_LAYERS, n_kv_heads=N_KV_HEADS, use_rope=true)
    cur_step = length(prefix)

    # -- Génération, un token à la fois via le cache incrémental --
    for step in 1:max_new_tokens
        nxt0 = argmax(logits_row) - 1
        if nxt0 == EOS_ID
            stopped_on_eos = true
            break
        end
        push!(gen0, nxt0)
        cur_step += 1
        logits_row = cached_step!(g, nxt0, cur_step)
    end
    dt = time() - t0
    reply = decode_ids(gen0)
    push!(HISTORY, Dict("role"=>"assistant", "content"=>reply))
    if verbose
        println("  [", length(gen0), " tokens, ", round(dt, digits=1), "s",
                 stopped_on_eos ? ", arrêt sur <|im_end|>" : ", tronqué à max_new_tokens", "]")
    end
    return reply
end

"""Vide l'historique de conversation -- à appeler pour repartir d'un échange neuf."""
reset_chat!() = (empty!(HISTORY); println("Historique vidé."))

println("chat(...) prêt -- préfixe en un passage batché, génération via cache KV.")

## 4. Essayer

Chaque cellule ci-dessous peut être ré-exécutée avec un nouveau texte. `reset_chat!()` avant de changer de sujet si vous ne voulez pas que l'ancien échange reste dans le contexte.

In [ ]:
reset_chat!()
t_cell8 = @elapsed rep_cell8 = chat("What is the capital of Egypt?")
println(rep_cell8)

In [ ]:
# Tour suivant DANS LA MÊME conversation -- teste si l'historique est vraiment réutilisé
println(chat("And what is a famous food from that city?"))

In [ ]:
# Nouvelle conversation, sujet différent
reset_chat!()
println(chat("Write a haiku about the ocean."))

In [ ]:
# Nouvelle conversation, sujet différent
reset_chat!()
println(chat("the minimum of  2,3,15,0"))

## 5. Comparaison contrôlée froid vs chaud, DANS CE NOTEBOOK (pas un script séparé)

Le coordinateur a signalé un désaccord réel : l'utilisateur a appelé `chat(...)` deux fois dans le même noyau et n'a observé "aucune vraie différence", alors que `kv_cache_chat_timing_probe_warm.jl` (un script séparé) montrait un facteur ~15x entre le premier et le deuxième appel. Les cellules 8/9 ci-dessus ne sont PAS une comparaison équitable : 7 tokens générés contre 37 -- le total brut (14.6s vs 21.7s) donne l'impression fausse d'un ralentissement, alors que normalisé par token c'est déjà ~3.8x plus rapide (2.09s/tok -> 0.59s/tok). Les deux cellules ci-dessous répètent EXACTEMENT le même prompt (même nombre de tokens de préfixe, réponse attendue identique en décodage glouton) pour une comparaison vraiment équitable, directement dans ce notebook, pas dans un script à part.

In [ ]:
# Appel A -- même prompt que la cellule 8, mais réexécuté ICI, plus tard dans
# la même session (kernel jamais redémarré depuis le début du notebook) --
# donc DÉJÀ chaud au sens du JIT (cellule 8 a déjà exercé ce chemin de calcul).
reset_chat!()
t_a = @elapsed rep_a = chat("What is the capital of Egypt?")
println("Appel A : ", round(t_a, digits=1), "s -- ", rep_a)

In [ ]:
# Appel B -- répète EXACTEMENT le même prompt une seconde fois (warm vs warm,
# stabilité) -- si l'appel A ci-dessus est déjà chaud, B doit être du même
# ordre de grandeur que A, pas plus lent.
reset_chat!()
t_b = @elapsed rep_b = chat("What is the capital of Egypt?")
println("Appel B : ", round(t_b, digits=1), "s -- ", rep_b)
println("\nComparaison directe (même prompt \"What is the capital of Egypt?\", même réponse attendue), temps TOTAL (tokenizer inclus) :")
println("  Cellule 8 (1er appel EVER de la session -- cache/JIT froid ET tokenizer froid) : ", round(t_cell8,digits=1), "s")
println("  Appel A ci-dessus (session déjà chaude depuis la cellule 8)                     : ", round(t_a,digits=1), "s")
println("  Appel B ci-dessus (encore un appel plus tard, même session)                     : ", round(t_b,digits=1), "s")

In [ ]:
using CUDA
GC.gc()       # Julia repère les tenseurs morts et les met à la poubelle
GC.gc()       # (Un 2ème passage est souvent utile pour les objets complexes)
CUDA.reclaim() # CUDA vide enfin la poubelle de la carte graphique

In [ ]:
# ==========================================
# CELLULE D'INTERFACE CHAT INTERACTIVE
# ==========================================
println("--- Démarrage du Chat Interactif avec Qwen2.5-1.5B (NeuroDSL) ---")
println("-> Tapez 'exit' ou 'quit' pour arrêter.")
println("-> Tapez 'reset' pour vider l'historique (repartir à zéro).")

# Amorçage à froid (pour payer le coût JIT de 14.2s une seule fois au lancement)
println("\n[Amorçage du moteur et du JIT CUDA en cours...]")
chat("Hello"; max_new_tokens=1, verbose=false)
reset_chat!()
println("[Moteur chaud et prêt. Temps de réponse attendu : ~4.1s]")

while true
    print("\n🧑 Vous : ")
    user_input = readline()
    
    if lowercase(strip(user_input)) in ["exit", "quit"]
        println("Fin de la session.")
        break
    elseif lowercase(strip(user_input)) == "reset"
        reset_chat!()
        continue
    elseif isempty(strip(user_input))
        continue
    end
    
    print("🤖 Qwen : ")
    # Appel de la fonction chat optimisée avec le cache KV -- IMPORTANT : chat()
    # RETOURNE le texte de la réponse (`return reply`), il ne l'affiche jamais
    # lui-même (verbose=false désactive même le résumé de timing). Sans
    # capturer et imprimer cette valeur de retour, la génération se déroule
    # bien en interne mais rien n'apparaît à l'écran -- c'est exactement le
    # bug qui donnait l'impression que le modèle "ne répondait pas".
    reply = chat(user_input; max_new_tokens=200, verbose=false)
    println(reply)
    println() # Saut de ligne pour la lisibilité
end